# sum-and-broadcast-duality — ex1: sum_back and broadcast_back as dual ops

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `sum-and-broadcast-duality`. Running the final beacon cell reports progress against the `Backprop: sum/broadcast duality` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: sum/broadcast duality` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sum-and-broadcast-duality`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sum-and-broadcast-duality"
DD_SUBTOPIC = "Backprop: sum/broadcast duality"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## sum/broadcast duality — quick refresher

Reduction and replication are **dual** under backprop. If the forward pass collapses an axis, the backward pass restores it; if the forward pass replicates an axis, the backward pass sums it back:

| forward op            | backward op                       |
|-----------------------|-----------------------------------|
| `out = x.sum(dim=k)`  | `grad_x = grad.unsqueeze(k).expand_as(x)` |
| `out = x.broadcast_to(big_shape)` | `grad_x = grad.sum_to(x.shape)` |

The math: `sum` is a linear map (matrix of all 1s along the axis). Its transpose is broadcast (matrix of all 1s the other way). Backprop sends gradients through the *transpose* of the forward linear map — so `sum` and `broadcast` swap roles.

Concretely for `sum_back(grad_out, out, x, *, dim, keepdim=False)`:
- If `keepdim=False` (axis was DROPPED), re-insert the axis:   `grad_out = grad_out.unsqueeze(dim)`.
- Then broadcast back to `x.shape`: `grad_x = grad_out.expand_as(x)`.

Symmetric: `broadcast_back(grad_out, out, x)` calls `unbroadcast(grad_out, x)` — sums out the axes that got expanded. Same operation seen from the other side of the duality.

### Exercise 1 — sum_back and broadcast_back as dual ops

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the sum/broadcast duality by writing sum_back (re-insert axis + expand) and broadcast_back (sum out expanded axes), demonstrating they are transposes of one another.
> Keywords: sum, broadcast, duality, back-fn, keepdim, unsqueeze
> ```

**KCs targeted:** `sum-and-broadcast-duality`, `unbroadcast-pattern`

Implement TWO dual back fns:

**1. `sum_back(grad_out, out, x, dim, keepdim=False)`** — backward for `out = x.sum(dim=dim, keepdim=keepdim)`.
   - If `keepdim=False`, `out` lost the axis at `dim`; re-insert it: `grad_out = grad_out.unsqueeze(dim)`.
   - Then broadcast to `x.shape`: `grad_x = grad_out.expand_as(x).clone()`.
   - (The `.clone()` matters — `expand_as` produces a view with stride 0, and downstream `+=` accumulations on views with overlapping memory misbehave. `.clone()` materializes a fresh contiguous tensor.)

**2. `broadcast_back(grad_out, out, x)`** — backward for `out = x.broadcast_to(out.shape)`. This is the `unbroadcast` pattern: sum out the axes that were expanded so `grad_x.shape == x.shape`.
   - Step A: while `grad_out.ndim > x.ndim`: `grad_out = grad_out.sum(dim=0)`.
   - Step B: for each axis `i` in `x.shape` where `x.shape[i] == 1` and `grad_out.shape[i] != 1`: `grad_out = grad_out.sum(dim=i, keepdim=True)`.

**Why this is the duality.** `sum` is a linear map (a matrix of ones along the summed axis). Its transpose is `broadcast` (ones the other way). Backprop sends gradients through the TRANSPOSE of the forward linear map. So `sum_back` ≈ broadcast, and `broadcast_back` ≈ sum. Same physical op, dual roles.

Inputs are plain `torch.Tensor`. No autograd. Return tensors with the right shape per back-fn convention.

In [ ]:
def sum_back(grad_out: Tensor, out: Tensor, x: Tensor, dim: int, keepdim: bool = False) -> Tensor:
    # If keepdim=False, the forward sum DROPPED the axis at `dim`.
    # Re-insert it (size 1) so we can expand cleanly back to x.shape.
    if not keepdim:
        grad_out = grad_out.unsqueeze(dim)
    # Broadcast back to x.shape. .clone() materializes a contiguous tensor
    # (expand_as gives a stride-0 view that misbehaves under accumulation).
    return grad_out.expand_as(x).clone()


def broadcast_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # Step A: peel leading axes that broadcasting added.
    while grad_out.ndim > x.ndim:
        grad_out = grad_out.sum(dim=0)
    # Step B: collapse size-1 axes that were expanded; keepdim preserves shape match.
    for i, size in enumerate(x.shape):
        if size == 1 and grad_out.shape[i] != 1:
            grad_out = grad_out.sum(dim=i, keepdim=True)
    return grad_out


<details><summary>Solution</summary>

```python
def sum_back(grad_out: Tensor, out: Tensor, x: Tensor, dim: int, keepdim: bool = False) -> Tensor:
    # If keepdim=False, the forward sum DROPPED the axis at `dim`.
    # Re-insert it (size 1) so we can expand cleanly back to x.shape.
    if not keepdim:
        grad_out = grad_out.unsqueeze(dim)
    # Broadcast back to x.shape. .clone() materializes a contiguous tensor
    # (expand_as gives a stride-0 view that misbehaves under accumulation).
    return grad_out.expand_as(x).clone()


def broadcast_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # Step A: peel leading axes that broadcasting added.
    while grad_out.ndim > x.ndim:
        grad_out = grad_out.sum(dim=0)
    # Step B: collapse size-1 axes that were expanded; keepdim preserves shape match.
    for i, size in enumerate(x.shape):
        if size == 1 and grad_out.shape[i] != 1:
            grad_out = grad_out.sum(dim=i, keepdim=True)
    return grad_out
```

**Why `unsqueeze + expand_as` and not just `expand_as`.** `expand_as` requires the source ndim to match the target ndim (or be smaller with appropriate trailing alignment). When `keepdim=False`, `grad_out` has one fewer dim than `x` at position `dim`. The `unsqueeze(dim)` puts the missing size-1 axis back so `expand_as` can do its work.

**Why `.clone()` after `expand_as`.** `expand_as` returns a VIEW with stride 0 along the expanded axis — every position in that axis points to the SAME memory cell. Doing `grad += something` on that view writes to the same cell N times, accumulating wrong. `.clone()` materializes a fresh contiguous tensor where each position has its own storage. Subtle bug — easy to miss until accumulation tests fail.

**The transpose intuition.** Think of `sum(dim=k)` as left-multiplying by a row vector of ones. Its transpose is a column vector of ones, which left-multiplied broadcasts. Backprop always sends gradients through the transpose of the forward linear op. So:
- forward = sum → backward = broadcast (insert + expand)
- forward = broadcast → backward = sum (collapse axes)

**Where this shows up in real models.** Every `(B, *)` mean / sum reduction in a loss function uses `sum_back` on the reverse pass. Every bias add (`out = x + bias` where `bias.shape == (C,)` and `x.shape == (B, C)`) broadcasts the bias forward and needs `broadcast_back` to compute `dL/dbias`. The duality is what makes broadcasting transparent for the model author.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()